# Assignment 1: Smart decisions with Jev (TypeSafe AI)

**Task:** route customer-support tickets (Russian / Kazakh / English / mixed) of a fictional Kazakhstani marketplace.
Jev answers three small typed questions; plain code combines them with thresholds. The same 80 hand-checked tickets go through
a big LLM (Claude Sonnet 5.5) asked for JSON, and we compare accuracy, latency, cost per 1,000 decisions and % invalid outputs.

> API key: put `OPENROUTER_API_KEY` in **Colab → Secrets** (key icon). Jev and Claude are both called through OpenRouter. The key is never written into this notebook.

In [ ]:
import os, sys
if 'google.colab' in sys.modules:
    %pip -q install typesafe-sdk openai
    from google.colab import userdata
    os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
    if not os.path.exists('jevlab.py'):
        # TODO: replace with your repo URL (or upload jevlab.py and data/ through the Colab file sidebar)
        !git clone https://github.com/YOUR_USER/YOUR_REPO.git repo && cp -r repo/* .
assert os.environ.get('OPENROUTER_API_KEY'), 'set OPENROUTER_API_KEY first'

import pandas as pd
import jevlab as J
os.makedirs('results', exist_ok=True)
pd.set_option('display.max_colwidth', 120, 'display.width', 200)

## 1. Workflow and typed questions

```mermaid
flowchart LR
    T[Ticket text] --> Q1[Choice: category<br/>billing / delivery / technical / account]
    T --> Q2[Score: urgency 0-2]
    T --> Q3[Noul: needs a human right now?]
    Q3 -->|P ≥ 0.5| H[Route: human]
    Q1 -->|P(escalate) < 0.5| R[Route: category queue]
    Q2 --> P[Priority tag: high if urgency = 2]
    R --> C{decision confidence ≥ threshold?}
    H --> C
    C -->|yes| A[Auto-route]
    C -->|no| M[Manual review]
```

Decision confidence = `P(escalate)` when escalating, otherwise `min(category confidence, 1 − P(escalate))`.

In [ ]:
gold = J.load_data('data/tickets.csv')
tricky = J.load_data('data/tricky.csv')
print(len(gold), 'tickets;', len(tricky), 'tricky cases')
print(gold.groupby(['lang']).size().to_dict(), gold['route'].value_counts().to_dict())
for name, q in J.jev_questions().items():
    print(name, '->', q)

## 2. Run Jev

In [ ]:
jev = J.run_jev(gold)               # one call per ticket, 3 typed questions each
jev.to_csv('results/jev_predictions.csv', index=False)
print(jev['valid'].mean(), 'valid;', jev['latency_s'].mean(), 's mean latency')
jev.head()

## 3. Run the LLM baselines (Claude asked for JSON)

In [ ]:
LLM_MODELS = ['anthropic/claude-sonnet-5.5']   # OpenRouter model id
llm = {m: J.run_llm(gold, m) for m in LLM_MODELS}
for m, df in llm.items():
    df.to_csv(f"results/{m.replace('/', '_')}_predictions.csv", index=False)
llm[LLM_MODELS[0]].head()

## 4. Results table: Jev vs LLM

In [ ]:
preds = {'Jev': jev, **llm}
table = J.results_table(gold, preds)
table.to_csv('results/results_table.csv')
show = table[['route_acc','category_acc','urgency_acc','escalate_recall','escalate_precision',
              'latency_mean_s','latency_p95_s','cost_per_1k_usd','invalid_pct_strict','invalid_pct']]
display(show.round(4))


## 5. Use the confidence

Each ticket gets a decision confidence. Everything at or above a threshold is auto-routed; everything below goes to a human.
For the LLMs the confidence is the number the model *says* in its JSON (verbalized confidence, not calibrated probabilities).

In [ ]:
summary = J.plot_confidence(gold, preds, 'results/confidence_plot.png', target=0.95)
for k, v in summary.items():
    print(k, v if v else 'never reaches 95% accuracy with at least 10 auto-decided cases')
from IPython.display import Image; Image('results/confidence_plot.png')

## 6. Break it: 10 tricky inputs

In [ ]:
tr_preds = {'Jev': J.run_jev(tricky), **{m: J.run_llm(tricky, m) for m in LLM_MODELS}}
report = J.tricky_report(tricky, tr_preds)
report.to_csv('results/tricky_cases.csv', index=False)
report[['id','trick_type','text','expected_route'] + [c for c in report if c.endswith(': output') or c.endswith('route correct')]]

## 7. Conclusion (write this yourself after looking at the numbers)

*3-5 sentences: when would you use Jev and when an LLM, and why. Use your own results table, the confidence plot and the tricky cases as evidence.*